<a href="https://colab.research.google.com/github/yassinmmohey/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassinmmohey/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 2  Refresh / Content Opportunity Scoring.**

Task type: **ranking / scoring**, not plain classification. The real decision is "which page
does an editor open first this week," under limited review capacity that's an ordered queue,
not a single yes/no call per page. A classifier's probability is one input to that ranking, but
the deliverable a human consumes is the order, so I'll frame and evaluate this as a ranking
problem (Precision@K).

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
initial_rows = len(df)

# same filter as scripts/01_prepare_features.py: enough evidence to score, page not brand new
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

print(f"initial_rows={initial_rows:,} -> prepared_rows={len(df):,} across {df['client_id'].nunique()} clients")


initial_rows=30,000 -> prepared_rows=30,000 across 32 clients


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [11]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"declining_rows={df['is_declining_label'].sum():,}  declining_rate={df['is_declining_label'].mean():.3f}")
print(df["trend_direction"].value_counts())

# guard rail: confirm the leak columns never enter the model feature list
LEAK_COLUMNS = {"trend_direction", "trend_pct"}
model_features = ["log_impressions_90d", "log_clicks_90d", "content_age_days", "days_since_last_update",
                   "ctr", "avg_position", "engagement_rate", "scroll_rate", "word_count", "has_word_count"]
assert LEAK_COLUMNS.isdisjoint(model_features), "label-derived column leaked into features!"
print("leak check passed: trend_direction / trend_pct excluded from model_features")


declining_rows=16,262  declining_rate=0.542
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
leak check passed: trend_direction / trend_pct excluded from model_features


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** The queue is read top-down by a reviewer with limited time (the guide's own
example capacity), so what matters is "of the top 50 pages I hand a reviewer, how many were
actually right" — that's exactly Precision@K, and it's the metric the guide's own baseline
(`0.240`) vs. random forest (`0.740`) comparison is built on. Generic accuracy would reward the
model for getting the easy 29,000 low-priority pages right and hide how it does on the 50 pages
that actually get read.

I'll compute Precision@50 myself below on a **client-holdout** test split, on a **non-circular** version
of the baseline: the guide's `declining_with_demand` rule reuses `trend_direction` directly,
which is the same information as the label, so I dropped it from the baseline score to keep this
a fair fixed-rule-vs-model comparison rather than comparing the label against itself.

In [12]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# has_-flags instead of blind fillna (skill gotcha: missingness follows content_type)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["word_count"] = df["word_count"].fillna(0)

def rule_score(r):
    # non-circular baseline: drops 'declining_with_demand' (it reuses trend_direction == label)
    s = 0
    s += int(r["days_since_last_update"] >= 180 and r["impressions_90d"] >= 500)             # stale_visible_page
    s += int(0 < r["word_count"] < 1200 and r["impressions_90d"] >= 250)                       # thin_visible_page
    s += int(0 < r["avg_position"] <= 10 and r["content_age_days"] >= 180)                     # page_one_decay_risk
    s += int(r["impressions_90d"] >= 500 and 0 < r["avg_position"] <= 20 and r["ctr"] < 0.5)   # low_ctr_visible_page
    s += int(r["sessions_90d"] >= 30 and (r["engagement_rate"] < 30 or r["scroll_rate"] < 30)) # low_engagement_visible_page
    return s

df["baseline_rule_score"] = df.apply(rule_score, axis=1)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
assert set(train["client_id"]).isdisjoint(set(test["client_id"])), "client leaked across split!"

for d in (train, test):
    d["log_impressions_90d"] = np.log1p(d["impressions_90d"])
    d["log_clicks_90d"] = np.log1p(d["clicks_90d"])

X_train, X_test = train[model_features].fillna(0), test[model_features].fillna(0)
y_train, y_test = train["is_declining_label"], test["is_declining_label"]

scaler = StandardScaler().fit(X_train)
clf = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
test["model_prob"] = clf.predict_proba(scaler.transform(X_test))[:, 1]

def precision_at_k(scores, labels, k=50):
    top_k = np.argsort(-scores)[:k]
    return labels.values[top_k].mean()

p50_rule  = precision_at_k(test["baseline_rule_score"].values, y_test, 50)
p50_model = precision_at_k(test["model_prob"].values, y_test, 50)
auc_rule  = roc_auc_score(y_test, test["baseline_rule_score"])
auc_model = roc_auc_score(y_test, test["model_prob"])

print(f"test set: n={len(test):,}, base decline rate={y_test.mean():.3f}, held-out clients={test['client_id'].nunique()}")
print(f"fixed rule  -> ROC AUC {auc_rule:.3f} | Precision@50 {p50_rule:.3f}")
print(f"log. model  -> ROC AUC {auc_model:.3f} | Precision@50 {p50_model:.3f}")


test set: n=10,834, base decline rate=0.559, held-out clients=10
fixed rule  -> ROC AUC 0.511 | Precision@50 0.440
log. model  -> ROC AUC 0.641 | Precision@50 0.820


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

one row = **one pseudonymized content item** (`content_id`), after keeping only pages with
`impressions_90d > 0` and `content_age_days >= 90` (enough evidence, not brand-new), deduped on
`content_id` per the skill's ID rule (join/group/split key only, never a feature). On the real
starter file this filter changes nothing (`30,000 -> 30,000` the starter export was already
pre-filtered), which itself is worth noting rather than assuming.

In [13]:
print(f"rows={len(df):,}  unique content_id={df['content_id'].nunique():,}  unique client_id={df['client_id'].nunique()}")
df[["content_id","client_id","content_type","impressions_90d","content_age_days",
    "avg_position","ctr","trend_direction","is_declining_label"]].head(5)


rows=30,000  unique content_id=30,000  unique client_id=32


,content_id,client_id,content_type,impressions_90d,content_age_days,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,187,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,445,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,141,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,463,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,263,44.0,0.13,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


 The fixed rule is barely better than random ranking on AUC (0.511 ~ coin flip) — it's a set of
hard OR'd thresholds, so two very different pages that each trip *one* flag score identically to
a page that's bad on every dimension; it can't weigh evidence. The model, given the same
underlying signals, learns how they trade off and interact continuously, and more than doubles
Precision@50 on clients it never saw in training.

This is also why a single if-statement isn't enough for this lane: the guide's five safe signals
(visibility, freshness, position, CTR, engagement) move at different rates, correlate with each
other in non-obvious ways, and their *relative* importance differs by content type exactly the
"too messy to hand-write" case the framing-ml-problems skill flags as where ML earns its place.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.